In [31]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv("../data/processed/telco_churn_clean.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  Paymen

In [34]:
x = df.drop(["customerID","Churn"], axis=1)
y = df["Churn"]

x.shape, y.shape

((7032, 19), (7032,))

In [35]:
y = y.map({
    "No":0,
    'Yes': 1
})

In [36]:
y.value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [37]:
numerical_cols = x.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols = x.select_dtypes(include=["object", "string", "str"]).columns.tolist()

In [38]:
numerical_cols

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [39]:
categorical_cols

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [40]:
numerical_cols.remove("SeniorCitizen")
categorical_cols.append("SeniorCitizen")

In [41]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=1, stratify=y)

In [42]:
x_train.shape, x_test.shape

((5625, 19), (1407, 19))

In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [44]:
num_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

cat_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore", drop='first'))]
)

In [45]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_cols),
        ("cat", cat_transformer, categorical_cols)
    ]
)

In [ ]:
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

In [49]:
x_train_processed.shape, x_test_processed.shape

((5625, 30), (1407, 30))

In [51]:
type(x_train_processed)

numpy.ndarray

In [56]:
from pathlib import Path
import joblib

processed_dir = Path("../data/processed")
models_dir = Path("../models")

processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(x_train_processed, processed_dir / "x_train_processed.pkl")
joblib.dump(x_test_processed, processed_dir / "x_test_processed.pkl")

joblib.dump(y_train, processed_dir / "y_train.pkl")
joblib.dump(y_test, processed_dir / "y_test.pkl")

joblib.dump(preprocessor, models_dir / "preprocessor.pkl")

['..\\models\\preprocessor.pkl']